# Florence-2 Fine-tuning on DocVQA

Notebook for Google Colab A100. Data is pulled from Hugging Face Hub via `src/colab_setup.py`, and checkpoints are uploaded back to Hugging Face Hub while metrics and tables are logged to Comet.

Prompt format follows the official Florence-2 model card pattern: `prompt = task_prompt + text_input`, then `processor(text=prompt, images=image, ...)`. For VQA, this notebook uses `task_prompt = "<VQA>"` and appends the question after it.

Reference: https://huggingface.co/microsoft/Florence-2-base

## 0. Setup

In [ ]:
%pip install -q huggingface_hub comet_ml python-dotenv timm einops

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'src').exists() and (candidate / 'notebooks').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    return None


def get_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)


project_root = find_project_root(Path.cwd())
repo_url = get_colab_secret('COURSE_WORK2026_REPO_URL')

if project_root is None:
    clone_dir = Path('/content/course_work2026')
    if not (clone_dir / '.git').exists():
        if not repo_url:
            raise RuntimeError('Could not detect the repository locally and COURSE_WORK2026_REPO_URL is not set in Colab secrets.')
        subprocess.run(['git', 'clone', repo_url, str(clone_dir)], check=True)
    project_root = clone_dir

sys.path.insert(0, str(project_root / 'src'))

from colab_setup import setup_colab

setup_summary, experiment = setup_colab(repo_url=repo_url)
experiment.set_name('florence2-finetune')

PROJECT_ROOT = Path(setup_summary['project_root'])
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
TRAIN_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'train_florence2.jsonl'
VALIDATION_JSONL = ARTIFACTS_ROOT / 'finetuning_generative' / 'validation_florence2.jsonl'
OUTPUT_ROOT = ARTIFACTS_ROOT / 'finetuning_generative' / 'florence2_base_colab'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
TRAIN_LOSS_CSV = OUTPUT_ROOT / 'train_loss.csv'
SANITY_RESULTS_CSV = OUTPUT_ROOT / 'sanity_check_results.csv'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(json.dumps(setup_summary, ensure_ascii=False, indent=2))
print({'project_root': str(PROJECT_ROOT), 'train_jsonl_exists': TRAIN_JSONL.exists(), 'validation_jsonl_exists': VALIDATION_JSONL.exists()})

## 1. Load Data

In [ ]:
import re
import time

import comet_ml
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoProcessor
from huggingface_hub import HfApi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
HF_CHECKPOINT_REPO = os.getenv('HF_MODEL_REPO') or 'karimkramin/docvqa-privacy-checkpoints'
api = HfApi()
api.create_repo(repo_id=HF_CHECKPOINT_REPO, repo_type='model', private=True, exist_ok=True)

QUESTION_RE = re.compile(r'<Question>(.*?)</Question>', flags=re.IGNORECASE | re.DOTALL)


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def extract_question(task_prompt: str) -> str:
    match = QUESTION_RE.search(task_prompt)
    if match:
        return match.group(1).strip()
    return task_prompt.replace('<VQA>', '').strip()


def resolve_image_path(raw_path: str, split: str) -> Path:
    original = Path(raw_path)
    if original.exists():
        return original
    benchmark_dir = 'benchmark_train' if split == 'train' else 'benchmark'
    candidate = ARTIFACTS_ROOT / 'docqa_recovery' / benchmark_dir / 'images' / 'original' / original.name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f'Cannot resolve image path for {raw_path}')


def prepare_florence_rows(rows: list[dict]) -> list[dict]:
    prepared = []
    for row in rows:
        question = extract_question(str(row['task_prompt']))
        image_path = resolve_image_path(str(row['image_path']), str(row['split']))
        prepared.append({
            'example_id': row['example_id'],
            'split': row['split'],
            'question': question,
            'task_prompt': '<VQA>',
            'task_prompt_with_question': f'<VQA>{question}',
            'answer': str(row['answer']).strip(),
            'image_path': str(image_path),
        })
    return prepared


train_rows = prepare_florence_rows(load_jsonl(TRAIN_JSONL))
validation_rows = prepare_florence_rows(load_jsonl(VALIDATION_JSONL))

processor = AutoProcessor.from_pretrained('microsoft/Florence-2-base', trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    'microsoft/Florence-2-base',
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)
model.config.use_cache = False

display(pd.DataFrame(train_rows[:3])[['example_id', 'question', 'task_prompt_with_question', 'answer', 'image_path']])

experiment.log_parameter('train_size', len(train_rows))
experiment.log_parameters({
    'validation_size': len(validation_rows),
    'model_name': 'microsoft/Florence-2-base',
    'task_prompt': '<VQA>',
    'checkpoint_repo': HF_CHECKPOINT_REPO,
})

print({'device': str(device), 'torch_dtype': str(torch_dtype), 'train_examples': len(train_rows), 'validation_examples': len(validation_rows)})

## 2. Dataset and DataLoader

In [ ]:
class FlorenceVQADataset(Dataset):
    def __init__(self, rows: list[dict], processor: AutoProcessor):
        self.rows = rows
        self.processor = processor
        self.tokenizer = processor.tokenizer

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> dict:
        row = self.rows[idx]
        image = Image.open(row['image_path']).convert('RGB')
        inputs = self.processor(
            images=image,
            text=row['task_prompt_with_question'],
            return_tensors='pt',
        )
        labels = self.tokenizer(
            row['answer'],
            return_tensors='pt',
            add_special_tokens=True,
        ).input_ids.squeeze(0)
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'labels': labels,
            'question': row['question'],
            'answer': row['answer'],
            'image_path': row['image_path'],
            'example_id': row['example_id'],
        }


def make_collate_fn(processor: AutoProcessor):
    tokenizer = processor.tokenizer
    pad_token_id = tokenizer.pad_token_id
    if pad_token_id is None:
        pad_token_id = tokenizer.eos_token_id

    def collate_fn(batch: list[dict]) -> dict:
        input_ids = torch.nn.utils.rnn.pad_sequence(
            [item['input_ids'] for item in batch],
            batch_first=True,
            padding_value=pad_token_id,
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [item['attention_mask'] for item in batch],
            batch_first=True,
            padding_value=0,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [item['labels'] for item in batch],
            batch_first=True,
            padding_value=-100,
        )
        pixel_values = torch.stack([item['pixel_values'] for item in batch])
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'pixel_values': pixel_values,
            'labels': labels,
            'question': [item['question'] for item in batch],
            'answer': [item['answer'] for item in batch],
            'image_path': [item['image_path'] for item in batch],
            'example_id': [item['example_id'] for item in batch],
        }

    return collate_fn


train_dataset = FlorenceVQADataset(train_rows, processor)
validation_dataset = FlorenceVQADataset(validation_rows, processor)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
    collate_fn=make_collate_fn(processor),
)

sample_batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in sample_batch.items() if hasattr(v, 'shape')})

## 3. Fine-tuning

In [ ]:
NUM_EPOCHS = 50
LEARNING_RATE = 5e-5
SAVE_EPOCHS = {1, 3, 10, 30, 50}

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
train_history = []


def save_checkpoint(model, processor, optimizer, epoch: int, mean_loss: float) -> Path:
    checkpoint_path = CHECKPOINT_DIR / f'epoch_{epoch}'
    checkpoint_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(checkpoint_path)
    processor.save_pretrained(checkpoint_path)
    torch.save(
        {
            'epoch': epoch,
            'optimizer_state_dict': optimizer.state_dict(),
            'mean_loss': mean_loss,
        },
        checkpoint_path / 'trainer_state.pt',
    )
    return checkpoint_path


for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_losses = []
    epoch_start = time.time()

    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS}', leave=False)
    for batch in progress:
        optimizer.zero_grad(set_to_none=True)

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        pixel_values = batch['pixel_values'].to(device=device, dtype=torch_dtype)
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(), dtype=torch.float16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels,
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = float(loss.detach().item())
        epoch_losses.append(loss_value)
        progress.set_postfix({'loss': f'{loss_value:.4f}'})

    epoch_seconds = time.time() - epoch_start
    mean_loss = sum(epoch_losses) / max(len(epoch_losses), 1)
    train_history.append({
        'epoch': epoch,
        'train_loss': mean_loss,
        'epoch_seconds': epoch_seconds,
    })
    pd.DataFrame(train_history).to_csv(TRAIN_LOSS_CSV, index=False)

    experiment.log_metric('train_loss', mean_loss, epoch=epoch)
    print(f'Epoch {epoch:02d} | loss={mean_loss:.4f} | time={epoch_seconds:.1f}s')

    if epoch in SAVE_EPOCHS:
        checkpoint_path = save_checkpoint(model, processor, optimizer, epoch, mean_loss)
        api.upload_folder(
            folder_path=str(checkpoint_path),
            repo_id=HF_CHECKPOINT_REPO,
            repo_type='model',
            path_in_repo=f'florence2/epoch_{epoch}',
        )
        experiment.log_metric('checkpoint_saved', epoch, epoch=epoch)
        print(f'  saved and uploaded checkpoint: {checkpoint_path}')

print(f'Train log saved to: {TRAIN_LOSS_CSV}')

## 4. Sanity Check

In [ ]:
def normalize_text(text: str) -> str:
    return ' '.join(str(text).strip().lower().split())


def sanitize_prediction_text(text: str) -> str:
    text = str(text or '')
    text = re.sub(r'<[^>]+>', ' ', text)
    text = ' '.join(text.replace('\n', ' ').split()).strip()
    return text


def exact_match(gold: str, pred: str) -> float:
    return float(normalize_text(gold) == normalize_text(pred))


def decode_prediction(processor, generated_ids, image: Image.Image, task_prompt: str) -> str:
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    if hasattr(processor, 'post_process_generation'):
        try:
            parsed = processor.post_process_generation(
                generated_text,
                task=task_prompt,
                image_size=(image.width, image.height),
            )
            if isinstance(parsed, dict):
                value = parsed.get(task_prompt)
                if isinstance(value, str):
                    return sanitize_prediction_text(value)
            elif isinstance(parsed, str):
                return sanitize_prediction_text(parsed)
        except Exception:
            pass
    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return sanitize_prediction_text(decoded)


def run_generation(model, processor, rows: list[dict], split_name: str, limit: int = 100) -> tuple[pd.DataFrame, float]:
    model.eval()
    results = []
    sample_rows = rows[:limit]

    for row in tqdm(sample_rows, desc=f'Sanity {split_name}'):
        image = Image.open(row['image_path']).convert('RGB')
        inputs = processor(
            images=image,
            text=row['task_prompt_with_question'],
            return_tensors='pt',
        )
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        pixel_values = inputs['pixel_values'].to(device=device, dtype=torch_dtype)

        with torch.no_grad():
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                max_new_tokens=50,
                num_beams=3,
                do_sample=False,
            )

        prediction = decode_prediction(processor, generated, image=image, task_prompt=row['task_prompt'])
        results.append({
            'split': split_name,
            'example_id': row['example_id'],
            'question': row['question'],
            'gold': row['answer'],
            'prediction': prediction,
            'exact_match': exact_match(row['answer'], prediction),
        })

    result_df = pd.DataFrame(results)
    em = float(result_df['exact_match'].mean()) if not result_df.empty else 0.0
    return result_df, em


epoch_50_dir = CHECKPOINT_DIR / 'epoch_50'
if not epoch_50_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found: {epoch_50_dir}')

sanity_processor = AutoProcessor.from_pretrained(epoch_50_dir, trust_remote_code=True)
sanity_model = AutoModelForCausalLM.from_pretrained(
    epoch_50_dir,
    torch_dtype=torch_dtype,
    trust_remote_code=True,
).to(device)
sanity_model.eval()

seen_results, em_seen = run_generation(sanity_model, sanity_processor, train_rows, 'seen_train', limit=100)
unseen_results, em_unseen = run_generation(sanity_model, sanity_processor, validation_rows, 'unseen_validation', limit=100)

all_results = pd.concat([seen_results, unseen_results], ignore_index=True)
all_results.to_csv(SANITY_RESULTS_CSV, index=False)

experiment.log_metric('em_seen', em_seen)
experiment.log_metric('em_unseen', em_unseen)
experiment.log_table('sanity_check.csv', all_results)

print({'em_seen': em_seen, 'em_unseen': em_unseen, 'sanity_results_csv': str(SANITY_RESULTS_CSV)})
display(all_results[['split', 'question', 'gold', 'prediction', 'exact_match']].head(10))

## 5. Finish

In [ ]:
experiment.end()
print('Comet experiment ended.')